# Evaluator No-Document Full Seattle Test

Loads pre-existing RLM output and GENIUS classified policies, filters Seattle, and runs `LEAPEvaluator.evaluate(..., source_document_path=None)`.

Because `source_document_path` is omitted, the evaluator uses the no-document grader path: one embedding pass plus one OpenAI grader call for each accepted match above the similarity threshold. This notebook now runs the full Seattle sets by default.

In [3]:
import os
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "optimization":
    REPO_ROOT = REPO_ROOT.parent

OPTIMIZATION_DIR = REPO_ROOT / "optimization"
if str(OPTIMIZATION_DIR) not in sys.path:
    sys.path.insert(0, str(OPTIMIZATION_DIR))

from evaluator import LEAPEvaluator

GENIUS_GT_PATH = Path("/Users/ziyadhassan/GENIUS/notebooks/outputs/all_cities_kept_classified_policies_final.csv")
RLM_OUTPUT_PATH = REPO_ROOT / "output" / "output_v3" / "Seattle_policies.csv"

CITY = "Seattle_WA_US"
GT_CITY_CONTAINS = "Seattle"
SAMPLE_SIZE = None  # None = full Seattle evaluation; integer = quick subset
SIMILARITY_THRESHOLD = 0.55
MODEL = "gpt-5.4"

assert GENIUS_GT_PATH.exists(), GENIUS_GT_PATH
assert RLM_OUTPUT_PATH.exists(), RLM_OUTPUT_PATH
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"

print("GT:", GENIUS_GT_PATH)
print("RLM:", RLM_OUTPUT_PATH)

GT: /Users/ziyadhassan/GENIUS/notebooks/outputs/all_cities_kept_classified_policies_final.csv
RLM: /Users/ziyadhassan/LEAP/output/output_v3/Seattle_policies.csv


In [4]:
gt_df = pd.read_csv(GENIUS_GT_PATH)
rlm_df = pd.read_csv(RLM_OUTPUT_PATH)

# Keep only GENIUS rows whose city label contains Seattle.
gt_city = gt_df.loc[
    gt_df["city"].astype(str).str.contains(GT_CITY_CONTAINS, case=False, na=False)
].copy()

# Align GENIUS field names with what evaluator.py reads.
if "secondary_category" not in gt_city.columns and "secondary_categories" in gt_city.columns:
    gt_city["secondary_category"] = gt_city["secondary_categories"]

if "source_quote" not in gt_city.columns and "verbatim_text" in gt_city.columns:
    gt_city["source_quote"] = gt_city["verbatim_text"]

required = ["policy_statement", "primary_category", "role", "parent_statement"]
gt_city = gt_city.dropna(subset=["policy_statement", "primary_category"])
rlm_df = rlm_df.dropna(subset=["policy_statement", "primary_category"])

print(f"GENIUS rows where city contains {GT_CITY_CONTAINS!r}: {len(gt_city)} policies")
print(f"Matched GT city values: {sorted(gt_city['city'].dropna().unique().tolist())}")
print(f"RLM extracted: {len(rlm_df)} policies")
display(gt_city[required + ["city", "is_financial_instrument"]].head(3))
display(rlm_df[required + ["financial_instrument"]].head(3))

GENIUS rows where city contains 'Seattle': 110 policies
Matched GT city values: ['Seattle_WA_US']
RLM extracted: 27 policies


,policy_statement,primary_category,role,parent_statement,city,is_financial_instrument
1746,Funding (Transportation Choices): secure adequ...,Mitigation,parent,NaN,Seattle_WA_US,yes
1747,Renew and extend the time frame of the Bridgin...,Mitigation,sub,Funding (Transportation Choices): secure adequ...,Seattle_WA_US,yes
1748,Secure local or transit agency authority to le...,Mitigation,sub,Funding (Transportation Choices): secure adequ...,Seattle_WA_US,yes


,policy_statement,primary_category,role,parent_statement,financial_instrument
0,Seattle commits to become carbon neutral by 2050.,Mitigation,individual,NaN,no
1,Seattle's right-of-way charging pilot permits ...,Mitigation,individual,NaN,no
2,Seattle City Light will install 20 public fast...,Mitigation,individual,NaN,no


In [5]:
# Full Seattle evaluation by default.
# WARNING: this makes one grader call per accepted match above SIMILARITY_THRESHOLD.
if SAMPLE_SIZE is None:
    extracted_policies = rlm_df.to_dict("records")
    ground_truth_policies = gt_city.to_dict("records")
    run_label = "FULL"
else:
    extracted_policies = rlm_df.head(SAMPLE_SIZE).to_dict("records")
    ground_truth_policies = gt_city.head(SAMPLE_SIZE).to_dict("records")
    run_label = f"HEAD({SAMPLE_SIZE})"

print(f"Run: {run_label}")
print(f"Evaluating {len(extracted_policies)} extracted vs {len(ground_truth_policies)} GT policies")
print(f"Similarity threshold: {SIMILARITY_THRESHOLD}")
print(f"Model: {MODEL}")

Evaluating 8 extracted vs 8 GT policies


In [6]:
DEFAULT_RUBRIC = (
    "Grade on specificity (quantified targets, deadlines, mechanisms), "
    "commitment strength (binding vs aspirational language), "
    "and accuracy relative to the source document."
)

evaluator = LEAPEvaluator(model=MODEL, similarity_threshold=SIMILARITY_THRESHOLD)

# No source_document_path on purpose: this exercises evaluator.py's no-document grader path.
result = evaluator.evaluate(
    location=CITY,
    extracted_policies=extracted_policies,
    ground_truth_policies=ground_truth_policies,
    rubric=DEFAULT_RUBRIC,
)

result

EvaluationOutput(location='Seattle_WA_US', scores={'Mitigation': -1.0, 'Adaptation': 0.0, 'Resource Efficiency': -1.0, 'Nature-Based Solutions': 0.0}, recall={'Mitigation': 0.0, 'Adaptation': 1.0, 'Resource Efficiency': 1.0, 'Nature-Based Solutions': 1.0}, fpr={'Mitigation': 1.0, 'Adaptation': 0.0, 'Resource Efficiency': 1.0, 'Nature-Based Solutions': 0.0}, grades={}, hierarchy_accuracy=0.0, composite_score=0.125, extraction_precision=0.0, extraction_recall=0.0, extraction_f1=0.0, role_agreement=0.0, parent_attribution_accuracy=1.0, primary_category_agreement=0.0, financial_instrument_agreement=0.0, secondary_category_agreement=0.0, plus_one_coverage=0.0, matched_count=0, unmatched_extracted_count=8, unmatched_ground_truth_count=8)

In [7]:
summary = {
    "composite_score": result.composite_score,
    "extraction_precision": result.extraction_precision,
    "extraction_recall": result.extraction_recall,
    "extraction_f1": result.extraction_f1,
    "role_agreement": result.role_agreement,
    "parent_attribution_accuracy": result.parent_attribution_accuracy,
    "primary_category_agreement": result.primary_category_agreement,
    "financial_instrument_agreement": result.financial_instrument_agreement,
    "secondary_category_agreement": result.secondary_category_agreement,
    "plus_one_coverage": result.plus_one_coverage,
    "matched_count": result.matched_count,
    "unmatched_extracted_count": result.unmatched_extracted_count,
    "unmatched_ground_truth_count": result.unmatched_ground_truth_count,
}

pd.Series(summary).to_frame("value")

,value
composite_score,0.125
extraction_precision,0.000
extraction_recall,0.000
extraction_f1,0.000
role_agreement,0.000
parent_attribution_accuracy,1.000
primary_category_agreement,0.000
financial_instrument_agreement,0.000
secondary_category_agreement,0.000
plus_one_coverage,0.000


In [8]:
pd.DataFrame({
    "score": result.scores,
    "recall": result.recall,
    "fpr": result.fpr,
})

,score,recall,fpr
Mitigation,-1.0,0.0,1.0
Adaptation,0.0,1.0,0.0
Resource Efficiency,-1.0,1.0,1.0
Nature-Based Solutions,0.0,1.0,0.0
